# lltPretrainData training runs (Run Library)

Two queues on Kaggle's 2x T4, one per GPU, over the token files from llt-ptd-prep.

In [ ]:

import glob, json, os, platform, subprocess, sys, time
T_SESSION = time.time()
print("python", platform.python_version())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"], capture_output=True, text=True).stdout)
hits = glob.glob("/kaggle/input/**/ptd_prep_stats.json", recursive=True)
print(hits)
DATA = os.path.dirname(hits[0])
print(sorted(os.listdir(DATA)))


In [ ]:
WORKER_SRC = '"""lltPretrainData training worker (Kaggle GPU kernel llt-ptd-train).\n\nRuns a queue of configs on ONE GPU (the launcher sets CUDA_VISIBLE_DEVICES before this process\nimports torch). The model is the lltTinyStories GPT (kaggle/worker_tinystories.py): pre-norm,\nRMSNorm, RoPE, SwiGLU, tied embeddings, fp16 autocast with GradScaler on a T4.\n\nUnlike the TinyStories worker, every run here reads a FIXED token stream in a fixed shuffled\nwindow order, so the number of times each document (and each canary copy) is seen is exact.\n\nUsage: python worker_ptd.py <queue.json> <data_dir> <out_dir>\n"""\nimport json, math, os, platform, sys, time\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nqueue_path, data_dir, out_dir = sys.argv[1:4]\nQUEUE = json.load(open(queue_path))\nDEV = "cuda" if torch.cuda.is_available() else "cpu"\nif DEV == "cpu":\n    torch.set_num_threads(2)\nCAP = torch.cuda.get_device_capability(0) if DEV == "cuda" else (0, 0)\n# Pick the dtype from the hardware, not from torch.cuda.is_bf16_supported(), which returns True on a T4.\nAMP_DTYPE = torch.bfloat16 if CAP >= (8, 0) else torch.float16\nUSE_SCALER = DEV == "cuda" and AMP_DTYPE == torch.float16\nSTATS = json.load(open(os.path.join(data_dir, "ptd_prep_stats.json")))\nEOT = STATS["meta"]["eot"]\nTOK_BYTES = np.load(os.path.join(data_dir, "token_bytes.npy"))\n_CACHE = {}\n\n\ndef corpus(name):\n    if name not in _CACHE:\n        _CACHE[name] = (np.load(os.path.join(data_dir, name + ".npy"), mmap_mode="r"),\n                        np.load(os.path.join(data_dir, name + "_offsets.npy")))\n    return _CACHE[name]\n\n\ndef docs_of(name, max_tokens=None, max_docs=None):\n    arr, off = corpus(name)\n    ends = np.append(off[1:], arr.size)\n    out, tot = [], 0\n    for s, e in zip(off, ends):\n        if max_docs is not None and len(out) >= max_docs:\n            break\n        if max_tokens is not None and tot >= max_tokens:\n            break\n        out.append(np.asarray(arr[s:e])); tot += e - s\n    return out\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps=1e-5):\n        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps\n    def forward(self, x):\n        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)\n\n\ndef rope_tables(T, hd, device):\n    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))\n    f = torch.outer(torch.arange(T, device=device).float(), inv)\n    return f.cos(), f.sin()\n\n\ndef apply_rope(x, cos, sin):\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)\n    return out.flatten(-2).type_as(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h):\n        super().__init__(); self.h = h\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.qkv = nn.Linear(d, 3 * d, bias=False); self.o = nn.Linear(d, d, bias=False)\n        hid = int(8 * d / 3 / 64 + 0.999) * 64\n        self.w1 = nn.Linear(d, hid, bias=False); self.w3 = nn.Linear(d, hid, bias=False); self.w2 = nn.Linear(hid, d, bias=False)\n    def forward(self, x, cos, sin):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))\n        h = self.n2(x)\n        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))\n\n\nclass GPT(nn.Module):\n    def __init__(self, V, d, L, H, ctx):\n        super().__init__()\n        self.emb = nn.Embedding(V, d); self.blocks = nn.ModuleList([Block(d, H) for _ in range(L)])\n        self.norm = RMSNorm(d); self.head = nn.Linear(d, V, bias=False); self.head.weight = self.emb.weight\n        self.H, self.ctx = H, ctx\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                std = 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02\n                nn.init.normal_(p, 0.0, std)\n    def forward(self, idx):\n        T = idx.shape[1]\n        cos, sin = rope_tables(T, self.emb.embedding_dim // self.H, idx.device)\n        x = self.emb(idx)\n        for b in self.blocks: x = b(x, cos, sin)\n        return self.head(self.norm(x))\n\n\ndef autocast():\n    return torch.autocast(DEV, dtype=AMP_DTYPE, enabled=DEV == "cuda")\n\n\ndef windows_from(stream, ctx):\n    n = (stream.size - 1) // ctx\n    return [stream[i * ctx:i * ctx + ctx + 1] for i in range(n)]\n\n\ndef build_stream(C, need_tokens, rng, record):\n    """Return (list of windows of ctx+1 tokens in training order, anneal windows or None)."""\n    ctx = C["ctx"]\n    kind = C["data"]["kind"]\n    if kind == "single":\n        arr, _ = corpus(C["data"]["corpus"])\n        unique = int(C["data"].get("uniqueTokens", need_tokens + 1))\n        assert arr.size >= unique, (C["data"]["corpus"], arr.size, unique)\n        base = np.asarray(arr[:unique])\n        wins = windows_from(base, ctx)\n        record["data"] = {"corpus": C["data"]["corpus"], "uniqueTokens": int(base.size), "windowsPerEpoch": len(wins)}\n        order, epochs = [], 0\n        while len(order) * 1 < need_tokens // ctx:\n            p = rng.permutation(len(wins)); order.extend(p.tolist()); epochs += 1\n        order = order[: need_tokens // ctx]\n        record["data"]["epochs"] = round(len(order) / len(wins), 3)\n        return [wins[i] for i in order]\n    if kind == "canary":\n        can = np.load(os.path.join(data_dir, "canaries.npy"))\n        levels, per = C["data"]["levels"], C["data"]["perLevel"]\n        docs = []\n        plan = []\n        k = 0\n        for lvl in levels:\n            for _ in range(per):\n                plan.append((k, lvl)); k += 1\n        assert k <= len(can), (k, len(can))\n        for idx, lvl in plan:\n            for _ in range(lvl):\n                docs.append(("c", idx))\n        n_can_tok = sum(lvl * 101 for _, lvl in plan)\n        base = docs_of(C["data"]["corpus"], max_tokens=need_tokens + 1 - n_can_tok + 5000)\n        docs += [("d", i) for i in range(len(base))]\n        order = rng.permutation(len(docs))\n        parts, counts = [], np.zeros(k, dtype=np.int64)\n        tot = 0\n        for j in order:\n            t, i = docs[j]\n            piece = np.append(can[i], np.uint16(EOT)) if t == "c" else base[i]\n            parts.append(piece); tot += piece.size\n        stream = np.concatenate(parts)[: need_tokens + 1]\n        # Count the canary copies that survive the cut whole (the tail of the shuffled stream is cut).\n        pos = 0\n        for j in order:\n            t, i = docs[j]\n            L = 101 if t == "c" else base[i].size\n            if pos + L <= stream.size and t == "c":\n                counts[i] += 1\n            pos += L\n        record["canaryPlan"] = [{"index": int(i), "planned": int(l), "seen": int(counts[i])} for i, l in plan]\n        record["data"] = {"corpus": C["data"]["corpus"], "canaryTokensPlanned": int(n_can_tok), "streamTokens": int(stream.size)}\n        wins = windows_from(stream, ctx)\n        return [wins[i] for i in rng.permutation(len(wins))]\n    if kind == "mixture":\n        total = need_tokens + 1\n        doms = C["data"]["domains"]  # {name: weight}\n        by = C["data"]["by"]\n        pieces = {}\n        if by == "tokens":\n            for name, w in doms.items():\n                pieces[name] = np.concatenate(docs_of(name, max_tokens=int(w * total) + 4 * ctx))[: int(w * total) + 2 * ctx]\n        else:\n            means = {}\n            for name in doms:\n                ds = docs_of(name, max_docs=4000)\n                means[name] = float(np.mean([d.size for d in ds]))\n            n_docs = 1.02 * total / sum(w * means[n] for n, w in doms.items())\n            for name, w in doms.items():\n                pieces[name] = np.concatenate(docs_of(name, max_docs=max(1, int(round(w * n_docs)))))\n        record["data"] = {"by": by, "weights": doms,\n                          "tokensPerDomain": {n: int(p.size) for n, p in pieces.items()},\n                          "tokenShare": {n: round(p.size / sum(q.size for q in pieces.values()), 4) for n, p in pieces.items()}}\n        wins = []\n        for p in pieces.values():\n            wins += windows_from(p, ctx)\n        assert len(wins) >= need_tokens // ctx, ("mixture too short", len(wins), need_tokens // ctx)\n        order = rng.permutation(len(wins))[: need_tokens // ctx]\n        return [wins[i] for i in order]\n    raise ValueError(kind)\n\n\ndef run_one(C):\n    seed = C["seed"]\n    torch.manual_seed(seed); np.random.seed(seed)\n    rng = np.random.default_rng(seed)\n    ctx, B, micro = C["ctx"], C["batch"], C.get("micro", C["batch"])\n    accum = B // micro\n    steps = C["steps"]\n    model = GPT(C["vocab"], C["d"], C["layers"], C["heads"], ctx).to(DEV)\n    n_params = sum(p.numel() for p in model.parameters())\n    R = {"runId": C["runId"], "module": "lltPretrainData", "tier": "T2", "label": C["label"],\n         "hardware": {"gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else None,\n                      "count": torch.cuda.device_count() if DEV == "cuda" else 0,\n                      "capability": list(CAP), "concurrent": "one run per GPU, two GPUs busy"},\n         "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n                 "ampDtype": str(AMP_DTYPE).replace("torch.", ""), "gradScaler": USE_SCALER},\n         "config": {k: v for k, v in C.items() if k not in ("runId", "label")}, "seed": seed, "params": n_params,\n         "metrics": [], "trainCurve": [], "date": time.strftime("%Y-%m-%d")}\n    t_build = time.time()\n    wins = build_stream(C, steps * B * ctx, rng, R)\n    anneal = None\n    if C.get("anneal"):\n        a_steps = int(round(steps * C["anneal"]["frac"]))\n        sub = dict(C, data={"kind": "single", "corpus": C["anneal"]["corpus"]})\n        anneal = build_stream(sub, a_steps * B * ctx, np.random.default_rng(seed + 7), {})\n        R["anneal"] = {"corpus": C["anneal"]["corpus"], "steps": a_steps}\n    R["streamBuildSec"] = round(time.time() - t_build, 1)\n    assert len(wins) >= steps * B or anneal is not None, (len(wins), steps * B)\n\n    def get_batch(step, k):\n        if anneal is not None and step >= steps - R["anneal"]["steps"]:\n            j = (step - (steps - R["anneal"]["steps"])) * B + k * micro\n            src = anneal\n        else:\n            j = step * B + k * micro\n            src = wins\n        xy = np.stack([np.asarray(w, dtype=np.int64) for w in src[j:j + micro]])\n        x = torch.from_numpy(xy[:, :-1]).to(DEV, non_blocking=True)\n        y = torch.from_numpy(xy[:, 1:]).to(DEV, non_blocking=True)\n        return x, y\n\n    @torch.no_grad()\n    def evaluate(name, max_tokens):\n        arr, _ = corpus(name)\n        n = min(arr.size, max_tokens + 1)\n        ws = windows_from(np.asarray(arr[:n]), ctx)\n        model.eval(); nll, ntok, nbytes = 0.0, 0, 0\n        for i in range(0, len(ws), 32):\n            xy = np.stack([w.astype(np.int64) for w in ws[i:i + 32]])\n            x = torch.from_numpy(xy[:, :-1]).to(DEV); y_np = xy[:, 1:]\n            with autocast():\n                logits = model(x)\n            nll += F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), torch.from_numpy(y_np).to(DEV).reshape(-1), reduction="sum").item()\n            ntok += y_np.size; nbytes += int(TOK_BYTES[y_np].sum())\n        model.train()\n        if ntok == 0:\n            return None\n        return {"loss": round(nll / ntok, 4), "bpb": round(nll / (math.log(2) * nbytes), 4), "tokens": ntok}\n\n    def lr_at(s):\n        peak, wu = C["lr"], C["warmup"]\n        if s < wu:\n            return peak * (s + 1) / wu\n        if C.get("schedule") == "wsd":\n            d0 = int(steps * (1 - C["decayFrac"]))\n            return peak if s < d0 else peak * max(0.0, (steps - s) / max(1, steps - d0))\n        p = (s - wu) / max(1, steps - wu)\n        return peak * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))\n\n    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]\n    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]\n    opt = torch.optim.AdamW([{"params": decay, "weight_decay": C["wd"]}, {"params": no_decay, "weight_decay": 0.0}],\n                            lr=C["lr"], betas=(0.9, 0.95), fused=DEV == "cuda")\n    scaler = torch.amp.GradScaler("cuda") if USE_SCALER else None\n    if DEV == "cuda":\n        torch.cuda.reset_peak_memory_stats()\n    evals = C.get("evals", [])\n    t0 = time.time(); run_loss = 0.0; run_n = 0\n    for step in range(steps + 1):\n        if step % C["eval_every"] == 0 or step == steps:\n            cap = C.get("evalTokensFinal", 600000) if step == steps else C.get("evalTokens", 131072)\n            m = {"step": step, "tokens": step * B * ctx, "lr": lr_at(min(step, steps - 1)), "elapsedSec": round(time.time() - t0, 1)}\n            for e in evals:\n                r = evaluate(e, cap)\n                if r:\n                    m[e] = r\n            if DEV == "cuda":\n                m["memGB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n            R["metrics"].append(m); print(C["runId"], json.dumps(m), flush=True)\n        if step == steps:\n            break\n        for g in opt.param_groups:\n            g["lr"] = lr_at(step)\n        opt.zero_grad(set_to_none=True)\n        tot = 0.0\n        for k in range(accum):\n            x, y = get_batch(step, k)\n            with autocast():\n                logits = model(x)\n                loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.reshape(-1)) / accum\n            (scaler.scale(loss) if scaler else loss).backward(); tot += loss.item()\n        if scaler:\n            scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), C["clip"])\n        if scaler:\n            scaler.step(opt); scaler.update()\n        else:\n            opt.step()\n        run_loss += tot; run_n += 1\n        if run_n == 25:\n            R["trainCurve"].append({"step": step + 1, "trainLoss": round(run_loss / 25, 4)}); run_loss = 0.0; run_n = 0\n    R["trainSec"] = round(time.time() - t0, 1)\n    if DEV == "cuda":\n        R["peakMemGB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n    if C.get("canaryEval"):\n        R["canaryResults"] = canary_eval(model, R)\n    if C.get("hellaswag"):\n        R["hellaswag"] = hellaswag(model, ctx, C.get("hellaswagMax"))\n    if C.get("trainSetEval"):\n        # Loss on the repeated training tokens themselves (first 600k), to show the train/val gap\n        w = wins[: 600000 // ctx]\n        model.eval(); nll, n = 0.0, 0\n        with torch.no_grad():\n            for i in range(0, len(w), 32):\n                xy = np.stack([np.asarray(v, dtype=np.int64) for v in w[i:i + 32]])\n                x = torch.from_numpy(xy[:, :-1]).to(DEV); y = torch.from_numpy(xy[:, 1:]).to(DEV)\n                with autocast():\n                    lg = model(x)\n                nll += F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y.reshape(-1), reduction="sum").item(); n += y.numel()\n        R["trainSetLoss"] = round(nll / n, 4)\n    with open(os.path.join(out_dir, C["runId"] + ".json"), "w") as f:\n        json.dump(R, f, indent=1)\n    print("WROTE", C["runId"], json.dumps(R["metrics"][-1])[:400], flush=True)\n    del model, opt\n    if DEV == "cuda":\n        torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef canary_eval(model, R):\n    """Greedy-decode 50 tokens after each canary\'s 50-token prefix (preceded by EOT, as in the\n    stream). Extraction = all 50 tokens match the canary exactly."""\n    can = torch.from_numpy(np.load(os.path.join(data_dir, "canaries.npy")).astype(np.int64)).to(DEV)\n    plan = R["canaryPlan"]\n    idx = [p["index"] for p in plan]\n    c = can[idx]\n    x = torch.cat([torch.full((c.size(0), 1), EOT, device=DEV, dtype=torch.long), c[:, :50]], 1)\n    model.eval()\n    for _ in range(50):\n        with autocast():\n            lg = model(x)[:, -1, :].float()\n        x = torch.cat([x, lg.argmax(-1, keepdim=True)], 1)\n    gen = x[:, 51:]\n    tgt = c[:, 50:100]\n    match = (gen == tgt)\n    exact = match.all(1).cpu().numpy()\n    frac = match.float().mean(1).cpu().numpy()\n    lead = []\n    for row in match.cpu().numpy():\n        k = 0\n        while k < 50 and row[k]:\n            k += 1\n        lead.append(k)\n    # Teacher-forced loss on the canary suffix\n    xin = torch.cat([torch.full((c.size(0), 1), EOT, device=DEV, dtype=torch.long), c[:, :99]], 1)\n    with autocast():\n        lg = model(xin).float()\n    nll = F.cross_entropy(lg[:, 50:].reshape(-1, lg.size(-1)), c[:, 50:].reshape(-1), reduction="none").view(c.size(0), -1).mean(1).cpu().numpy()\n    model.train()\n    out = {}\n    for p, e, f_, l_, n_ in zip(plan, exact, frac, lead, nll):\n        d = out.setdefault(str(p["planned"]), {"n": 0, "exact": 0, "tokenMatch": [], "lead": [], "suffixLoss": [], "seen": []})\n        d["n"] += 1; d["exact"] += int(e); d["tokenMatch"].append(round(float(f_), 3)); d["lead"].append(int(l_))\n        d["suffixLoss"].append(round(float(n_), 3)); d["seen"].append(p["seen"])\n    for d in out.values():\n        d["extractionRate"] = round(d["exact"] / d["n"], 4)\n        d["meanTokenMatch"] = round(float(np.mean(d["tokenMatch"])), 4)\n        d["meanSuffixLoss"] = round(float(np.mean(d["suffixLoss"])), 4)\n    print("CANARY", R["runId"], {k: (v["extractionRate"], v["meanTokenMatch"]) for k, v in out.items()}, flush=True)\n    return out\n\n\n@torch.no_grad()\ndef hellaswag(model, ctx, limit=None):\n    items = json.load(open(os.path.join(data_dir, "hellaswag_val_tok.json")))[:limit]\n    model.eval(); correct = correct_sum = 0\n    for it in items:\n        seqs, spans = [], []\n        for e in it["endings"]:\n            s = (it["ctx"] + e)[-ctx:]\n            seqs.append(s); spans.append(len(e) if len(e) < len(s) else len(s) - 1)\n        L = max(len(s) for s in seqs)\n        x = torch.zeros((4, L), dtype=torch.long, device=DEV)\n        for i, s in enumerate(seqs):\n            x[i, :len(s)] = torch.tensor(s, device=DEV)\n        with autocast():\n            lp = F.log_softmax(model(x).float(), -1)\n        scores, sums = [], []\n        for i, s in enumerate(seqs):\n            n = spans[i]; T = len(s)\n            tgt = torch.tensor(s[T - n:], device=DEV)\n            v = lp[i, T - n - 1:T - 1].gather(1, tgt[:, None]).sum().item()\n            sums.append(v); scores.append(v / n)\n        correct += int(np.argmax(scores) == it["label"]); correct_sum += int(np.argmax(sums) == it["label"])\n    model.train()\n    n = len(items)\n    acc = correct / n\n    return {"items": n, "accNormByToken": round(acc, 4), "accSum": round(correct_sum / n, 4),\n            "ci95": round(1.96 * math.sqrt(acc * (1 - acc) / n), 4)}\n\n\nfor C in QUEUE:\n    try:\n        run_one(C)\n    except Exception as e:  # one failed config must not cost the rest of the queue\n        import traceback; traceback.print_exc()\n        with open(os.path.join(out_dir, C["runId"] + ".error.txt"), "w") as f:\n            f.write(repr(e))\n        if DEV == "cuda":\n            torch.cuda.empty_cache()\nprint("QUEUE DONE", flush=True)\n'
open('/tmp/worker.py', 'w').write(WORKER_SRC)


In [ ]:

# Check the notebook's language-ID call on Linux with fasttext-numpy2 (the lab page's step 2 code path).
import urllib.request
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fasttext-numpy2"], capture_output=True, text=True)
    print("pip fasttext-numpy2 exit", r.returncode)
    import fasttext
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "/tmp/lid.176.bin")
    lid = fasttext.load_model("/tmp/lid.176.bin")
    labels, probs = lid.predict("The library opens at nine every morning and closes at six.", k=1)
    print("fasttext-numpy2 predict", labels, probs, type(probs))
    json.dump({"labels": list(labels), "probs": [float(p) for p in probs]}, open("/kaggle/working/fasttext_numpy2_check.json", "w"))
except Exception as e:
    print("fasttext check failed", repr(e))


In [ ]:
QUEUES = [[{'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'U_train'}, 'evals': ['F_val', 'U_val', 'U_val_clean', 'U_val_leak', 'wiki_test', 'ts_val'], 'hellaswag': True, 'runId': 'lltPretrainData-T2-unfiltered-s0', 'label': 'ablation:data=unfiltered'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train'}, 'evals': ['F_val', 'U_val', 'U_val_clean', 'U_val_leak', 'wiki_test', 'ts_val'], 'hellaswag': True, 'runId': 'lltPretrainData-T2-filtered-s0', 'label': 'baseline'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'E_train'}, 'evals': ['F_val', 'U_val', 'U_val_clean', 'U_val_leak', 'wiki_test', 'ts_val'], 'hellaswag': True, 'runId': 'lltPretrainData-T2-edu-s0', 'label': 'ablation:data=classifier-top-half'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 1, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'U_train'}, 'evals': ['F_val', 'U_val', 'U_val_clean', 'U_val_leak', 'wiki_test', 'ts_val'], 'hellaswag': True, 'runId': 'lltPretrainData-T2-unfiltered-s1', 'label': 'ablation:data=unfiltered'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 1, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train'}, 'evals': ['F_val', 'U_val', 'U_val_clean', 'U_val_leak', 'wiki_test', 'ts_val'], 'hellaswag': True, 'runId': 'lltPretrainData-T2-filtered-s1', 'label': 'baseline'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'schedule': 'wsd', 'decayFrac': 0.2, 'data': {'kind': 'single', 'corpus': 'F_train'}, 'anneal': {'corpus': 'F_train', 'frac': 0.2}, 'evals': ['F_val', 'wiki_test', 'U_val'], 'runId': 'lltPretrainData-T2-wsd-anneal-F', 'label': 'ablation:anneal=F'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'schedule': 'wsd', 'decayFrac': 0.2, 'data': {'kind': 'single', 'corpus': 'F_train'}, 'anneal': {'corpus': 'E_train', 'frac': 0.2}, 'evals': ['F_val', 'wiki_test', 'U_val'], 'runId': 'lltPretrainData-T2-wsd-anneal-E', 'label': 'ablation:anneal=E'}], [{'vocab': 4096, 'd': 192, 'layers': 5, 'heads': 6, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'canary', 'corpus': 'F_train', 'levels': [0, 1, 10, 100, 1000], 'perLevel': 32}, 'canaryEval': True, 'evals': ['F_val'], 'runId': 'lltPretrainData-T2-canary-3m', 'label': 'experiment:canary,size=3M'}, {'vocab': 4096, 'd': 512, 'layers': 9, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 800, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'canary', 'corpus': 'F_train', 'levels': [0, 1, 10, 100, 1000], 'perLevel': 32}, 'canaryEval': True, 'evals': ['F_val'], 'micro': 32, 'runId': 'lltPretrainData-T2-canary-30m', 'label': 'experiment:canary,size=30M'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train', 'uniqueTokens': 19660801}, 'evals': ['F_val'], 'trainSetEval': True, 'runId': 'lltPretrainData-T2-epochs-1', 'label': 'ablation:epochs=1'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train', 'uniqueTokens': 9830401}, 'evals': ['F_val'], 'trainSetEval': True, 'runId': 'lltPretrainData-T2-epochs-2', 'label': 'ablation:epochs=2'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train', 'uniqueTokens': 4915201}, 'evals': ['F_val'], 'trainSetEval': True, 'runId': 'lltPretrainData-T2-epochs-4', 'label': 'ablation:epochs=4'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 600, 'eval_every': 200, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'single', 'corpus': 'F_train', 'uniqueTokens': 2457601}, 'evals': ['F_val'], 'trainSetEval': True, 'runId': 'lltPretrainData-T2-epochs-8', 'label': 'ablation:epochs=8'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 300, 'eval_every': 100, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'mixture', 'by': 'tokens', 'domains': {'F_train': 0.3333333333333333, 'wiki_train': 0.3333333333333333, 'ts_train': 0.3333333333333333}}, 'evals': ['F_val', 'wiki_test', 'ts_val'], 'runId': 'lltPretrainData-T2-mix-equal-tokens', 'label': 'ablation:mixture=equal-by-tokens'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 300, 'eval_every': 100, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'mixture', 'by': 'docs', 'domains': {'F_train': 0.3333333333333333, 'wiki_train': 0.3333333333333333, 'ts_train': 0.3333333333333333}}, 'evals': ['F_val', 'wiki_test', 'ts_val'], 'runId': 'lltPretrainData-T2-mix-equal-docs', 'label': 'broken:weights-by-documents'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 300, 'eval_every': 100, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'mixture', 'by': 'tokens', 'domains': {'F_train': 0.6, 'wiki_train': 0.2, 'ts_train': 0.2}}, 'evals': ['F_val', 'wiki_test', 'ts_val'], 'runId': 'lltPretrainData-T2-mix-web60', 'label': 'ablation:mixture=web60'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 300, 'eval_every': 100, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'mixture', 'by': 'tokens', 'domains': {'F_train': 0.2, 'wiki_train': 0.6, 'ts_train': 0.2}}, 'evals': ['F_val', 'wiki_test', 'ts_val'], 'runId': 'lltPretrainData-T2-mix-wiki60', 'label': 'ablation:mixture=wiki60'}, {'vocab': 4096, 'd': 320, 'layers': 8, 'heads': 8, 'ctx': 512, 'batch': 64, 'lr': 0.001, 'warmup': 100, 'steps': 300, 'eval_every': 100, 'wd': 0.1, 'clip': 1.0, 'seed': 0, 'evalTokens': 131072, 'evalTokensFinal': 600000, 'data': {'kind': 'mixture', 'by': 'tokens', 'domains': {'F_train': 0.2, 'wiki_train': 0.2, 'ts_train': 0.6}}, 'evals': ['F_val', 'wiki_test', 'ts_val'], 'runId': 'lltPretrainData-T2-mix-stories60', 'label': 'ablation:mixture=stories60'}]]

procs = []
for gpu, q in enumerate(QUEUES):
    qp = f"/tmp/queue{gpu}.json"; json.dump(q, open(qp, "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/worker{gpu}.log", "w")
    procs.append((subprocess.Popen([sys.executable, "/tmp/worker.py", qp, DATA, "/kaggle/working"], env=env, stdout=log, stderr=subprocess.STDOUT), log))
while any(p.poll() is None for p, _ in procs):
    time.sleep(60)
    for gpu in range(2):
        lines = open(f"/kaggle/working/worker{gpu}.log").read().splitlines()
        print(gpu, lines[-1][:300] if lines else "")
for p, log in procs:
    log.close(); print("exit", p.returncode)
print("session wall", round(time.time() - T_SESSION, 1))
for gpu in range(2):
    txt = open(f"/kaggle/working/worker{gpu}.log").read()
    print("=== worker", gpu, "tail"); print(txt[-4000:])
